In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from glob import glob as gb
import os
import re
import warnings
warnings.filterwarnings("ignore")

In [ ]:
############################
# CONFIGURACION (igual que en 00_procesa.sh / 01_uncertainty.py / 02_grafica.py)
var = 'tas'        # 'pr' o 'tas'
tri = 'jja'        # 'djf', 'mam', 'jja', 'son' o 'ann'
idir = 'data'      # carpeta de entrada (una subcarpeta por trimestre: data/<tri>/)
odir = 'out'       # carpeta de resultados (.nc)
fdir = 'figures'   # carpeta de figuras (.png)
vi_stat = 'var'    # variabilidad interna: 'var' (recomendado) o 'std' (metodo anterior)
ref_ini, ref_end = 1981, 2010  # periodo de referencia
fut_ini = 2020                 # primer año de resultados (si los datos lo permiten)
# trimestre -> prefijo de archivo
per = {'djf': 'm1', 'mam': 'm4', 'jja': 'm7', 'son': 'm10', 'ann': 'ann'}[tri]
print(var, per, 'VI:', vi_stat)

In [ ]:
shp_peru = gpd.read_file('shape/peru.shp')
shp_suda = gpd.read_file('shape/suda.shp')

In [ ]:
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['DejaVu Serif'], 'font.size': 12})

In [ ]:
# ESCENARIOS Y PERIODO COMUN #########################
# escenarios detectados de los nombres de archivo (..._ssp126.nc -> '126')
all_files = gb(os.path.join(idir, tri, '%s_%s_*_ssp*.nc' % (per, var)))
scenarios = sorted({m.group(1) for m in (re.search(r'_ssp(\d+)\.nc$', f) for f in all_files) if m})
print('Escenarios:', ', '.join('SSP' + e for e in scenarios))
# años de cada archivo; se usa el periodo comun a todos
years_file = {}
for file in sorted(all_files):
    yrs = xr.open_dataset(file)['time.year'].values
    years_file[file] = (int(yrs.min()), int(yrs.max()))
y_start = max(a for a, b in years_file.values())
y_stop = min(b for a, b in years_file.values())
print('Periodo comun: %d-%d' % (y_start, y_stop))
for file, (a, b) in years_file.items():
    if (a, b) != (y_start, y_stop): print('  aviso: %s tiene %d-%d' % (os.path.basename(file), a, b))
# VI usa ventanas de 30 años y luego media movil de 10 -> primer valor en y_start+39
a_ini = max(fut_ini, y_start + 39)
print('Periodo de resultados: %d-%d' % (a_ini, y_stop))

In [ ]:
# FUNCIONES #########################
def polinom(d, var):
    coef = d.polyfit(dim='time', deg=4, full=True) # genera los coeficientes del polinomio 4°
    s_poli = xr.polyval(coord=d.time, coeffs=coef['%s_polyfit_coefficients' % var]) # Serie polinomica
    residuo0 = ((d[var] / s_poli) * 100) - 100 # residuo = ε(t)
    prom_PR = s_poli.sel(time=slice(str(ref_ini), str(ref_end))).mean(dim='time') # promedio del Periodo de Referencia
    if 't' in var: delta0 = s_poli - ((s_poli / s_poli) * prom_PR) # delta = ΔT
    else: delta0 = ((s_poli / prom_PR) * 100) - 100 # delta = ΔP
    return delta0, residuo0

In [ ]:
########## PARTE 1 #########
j=0; k=0
for n, exp in enumerate(scenarios):
    files = gb(os.path.join(idir, tri, '%s_%s_*_ssp%s.nc' % (per, var, exp)))
    i = 0
    print('SSP%s (%d modelos)' % (exp, len(files)))
    for file in files[:]:
        ds0 = xr.open_dataset(file)
        ds = ds0[[var]].sel(time=slice(str(y_start), str(y_stop))) # solo el periodo comun
        ds = ds.sortby('lat', ascending=True) # latitud de sur a norte
        delta, residuo = polinom(ds, var)
        if i==0: delta2 = delta
        else: delta2 = xr.concat([delta, delta2], dim='time')
        if j==0: delta3 = delta; residuo4 = residuo
        else:
            delta3 = xr.concat([delta, delta3], dim='time')
            residuo4 = xr.concat([residuo, residuo4], dim='time')
        j=j+1; k=k+1; i=i+1
        #
    delta21 = delta2.sortby('time', ascending=True)
    var_delta21 = delta21.groupby('time.year').var() # var de todos los modelos en un escenario
    prom_delta21 = delta21.groupby('time.year').mean() # prom de todos los modelos en un escenario
    if n==0: lst_var_mdl = var_delta21; lst_prom_mdl = prom_delta21
    else:
        lst_var_mdl = xr.concat([lst_var_mdl, var_delta21], dim='year')
        lst_prom_mdl = xr.concat([lst_prom_mdl, prom_delta21], dim='year')

In [ ]:
########## PARTE 2 #########        
var_mdl_2=lst_var_mdl.sortby('year',ascending=True)
prom_mdl_2=lst_prom_mdl.sortby('year',ascending=True)
delta31=delta3.sortby('time',ascending=True)
residuo41=residuo4.sortby('time',ascending=True)

In [ ]:
#
G0 = delta31.groupby('time.year').mean() # promedio de todos los modelos en todos los escenarios
G0['year'] = pd.date_range(str(y_start), freq='12M', periods=len(G0.year))
G0 = G0.rename({'year': 'time'})
#incertidumbre de escenarios
SU0 = prom_mdl_2.groupby('year').var() # Scenario Uncertainty "SU" desde el var
SU0['year'] = pd.date_range(str(y_start), freq='12M', periods=len(SU0.year))
SU0 = SU0.rename({'year': 'time'})
#incertidumbre de modelos
MU0 = var_mdl_2.groupby('year').mean() # Model Uncertainty  "MU"
MU0['year'] = pd.date_range(str(y_start), freq='12M', periods=len(MU0.year))
MU0 = MU0.rename({'year': 'time'})

In [ ]:
########## PARTE 3 #########
year_c = y_start + 29; l=0 # ultimo año de la primera ventana de 30 años
for ktime in range(len(ds['time'])-30):
    y_i = residuo41['time.year'][ktime*k].values; yi = str(y_i.tolist())
    y_f = (residuo41['time.year'][ktime*k].values)+29; yf = str(y_f.tolist())
    residuo42 = residuo41.sel(time=slice(yi, yf))
    residuo43_std = residuo42.std(dim='time')
    residuo43_var = residuo42.var(dim='time')
    year_c = year_c+1
    #expande dimensiones: dim=tiempo
    time_d = pd.date_range('%d' % year_c, freq='1Y', periods=1)
    time_ds = xr.DataArray(time_d, [('time', time_d)])
    residuo44_std = residuo43_std.expand_dims(time=time_ds)
    residuo44_var = residuo43_var.expand_dims(time=time_ds)
    if l==0:
        residuo_45_std = residuo44_std
        residuo_45_var = residuo44_var
    else:
        residuo_45_std = xr.concat([residuo44_std, residuo_45_std], dim='time')
        residuo_45_var = xr.concat([residuo44_var, residuo_45_var], dim='time')
    l=l+1

In [ ]:
########## PARTE 4 #########
# vi_stat='var': variabilidad interna como varianza, asi TOE=G/VI**0.5 es senal/desviacion estandar
# y VI se suma con MU y SU (que tambien son varianzas). vi_stat='std' es el metodo anterior
if vi_stat == 'var': VI0 = residuo_45_var.sortby('time', ascending=True)
else: VI0 = residuo_45_std.sortby('time', ascending=True)
VI0['time'] = pd.date_range(str(y_start + 30), freq='12M', periods=len(VI0.time))
VI1 = VI0.rolling(time=10, center=False).mean()
# seleccionar el mismo periodo de datos
VI = VI1.sel(time=slice(str(a_ini), str(y_stop)))

In [ ]:
G = G0.sel(time=slice(str(a_ini), str(y_stop)))
SU = SU0.sel(time=slice(str(a_ini), str(y_stop)))
MU = MU0.sel(time=slice(str(a_ini), str(y_stop)))

In [ ]:
# calculo de STN y TOE
TU = VI+MU+SU
STN = G/(TU)**0.5
TOE = G/(VI)**0.5
########## PARTE 5 #########
os.makedirs(odir, exist_ok=True)
TOE.to_netcdf(os.path.join(odir, 'toe_%s_%s_%s.nc' % (var, per, vi_stat)))
MU.to_netcdf(os.path.join(odir, 'Tmu_%s_%s_%s.nc' % (var, per, vi_stat)))
SU.to_netcdf(os.path.join(odir, 'Tsu_%s_%s_%s.nc' % (var, per, vi_stat)))
VI.to_netcdf(os.path.join(odir, 'Tvi_%s_%s_%s.nc' % (var, per, vi_stat)))
print('Resultados: %d-%d' % (a_ini, y_stop))

In [ ]:
############################
# GRAFICOS (02_grafica.py): lee los resultados guardados en odir
ofile = os.path.join(odir, 'toe_%s_%s_%s.nc' % (var, per, vi_stat))
TOE = xr.open_dataset(ofile)['__xarray_dataarray_variable__']
y_ini = int(TOE['time.year'][0])  # primer y ultimo año de los resultados
y_end = int(TOE['time.year'][-1])
levels = list(range(y_ini, y_end + 2))                       # un color por año
year_ticks = list(range(y_ini, y_end - 4, 10)) + [y_end]     # etiquetas cada 10 años
cbar_kw = {'label': 'Year', 'shrink': 0.75,                  # barra al 75% del panel
           'ticks': [t + 0.5 for t in year_ticks], 'format': lambda x, pos: '%d' % x}
os.makedirs(fdir, exist_ok=True)

In [ ]:
t_TOE = TOE*np.nan
tpr = y_ini

In [ ]:
for tpru in range(len(TOE.time)):
    t_TOE=t_TOE.where(~((TOE<tpr)&(TOE['time.year']==tpr)),tpr)
    tpr=tpr+1

In [ ]:
if var=='pr':
    for umbr in [-1, 1]:
        TOE_umbr=TOE[0]*0
        for t in range(len(TOE.time)):
            if umbr==-1:
                TOE_umbr=TOE_umbr.where(~((TOE[t]<umbr)&(TOE_umbr<=0)),t_TOE[t])
                TOE_umbralnegativo_pr=TOE_umbr
            else:
                TOE_umbr=TOE_umbr.where(~((TOE[t]>umbr)&(TOE_umbr<=0)),t_TOE[t])
                TOE_umbralpositivo_pr=TOE_umbr
    TOE_1=TOE_umbralnegativo_pr.where(~(TOE_umbralnegativo_pr==0),np.nan)
    TOE_2=TOE_umbralpositivo_pr.where(~(TOE_umbralpositivo_pr==0),np.nan)
else:
    for umbr in [1, 2]:
        TOE_umbr=TOE[0]*0
        for t in range(len(TOE.time)):
            TOE_umbr=TOE_umbr.where(~((TOE[t]>umbr)&(TOE_umbr<=0)),t_TOE[t])
            if umbr==1: TOE_u1_tas=TOE_umbr
            else: TOE_u2_tas=TOE_umbr
    TOE_1=TOE_u1_tas.where(~(TOE_u1_tas==0),np.nan)
    TOE_2=TOE_u2_tas.where(~(TOE_u2_tas==0),np.nan)

In [ ]:
# nombres y titulos de los umbrales (señal/ruido)
if var == 'pr':
    name_2, title_2 = 'umbral_positivo', 'Aumento (+1)'
    name_1, title_1 = 'umbral_negativo', 'Disminución (-1)'
else:
    name_2, title_2 = 'umbral_alto', 'Umbral alto (+2)'
    name_1, title_1 = 'umbral_bajo', 'Umbral bajo (+1)'
fig, ax = plt.subplots(figsize=(10, 8))
TOE_2.plot(ax=ax, levels=levels, cmap='turbo', cbar_kwargs=cbar_kw)
# --- LIMITES GEOGRAFICOS ---
# Ajusta los limites de longitud (x) y latitud (y) segun tu region de interes
ax.set_xlim(-83, -67)   # Longitud
ax.set_ylim(-19, 0.5)   # Latitud
# --- SHAPEFILES ---
shp_peru.plot(ax=ax, fc='none', edgecolor='black', linewidth=0.8)
shp_suda.plot(ax=ax, fc='white', edgecolor='gray', linewidth=0.3)
# --- ETIQUETAS ---
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_title('Time of Emergence - %s (%s, %s, %d-%d, VI: %s)' % (title_2, var.upper(), tri.upper(), y_ini, y_end, vi_stat))
plt.tight_layout()
plt.savefig(os.path.join(fdir, 'TOE_%s_%s_%s_%s.png' % (var, tri, name_2, vi_stat)), dpi=100, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
TOE_1.plot(ax=ax, levels=levels, cmap='turbo', cbar_kwargs=cbar_kw)
ax.set_xlim(-83, -67)
ax.set_ylim(-19, 0.5)
shp_peru.plot(ax=ax, fc='none', edgecolor='black', linewidth=0.8)
shp_suda.plot(ax=ax, fc='white', edgecolor='gray', linewidth=0.3)
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
ax.set_title('Time of Emergence - %s (%s, %s, %d-%d, VI: %s)' % (title_1, var.upper(), tri.upper(), y_ini, y_end, vi_stat))
plt.tight_layout()
plt.savefig(os.path.join(fdir, 'TOE_%s_%s_%s_%s.png' % (var, tri, name_1, vi_stat)), dpi=100, bbox_inches='tight')

### FRACCION

In [ ]:
varn = '__xarray_dataarray_variable__'
mu_perc = xr.open_dataset(os.path.join(odir, 'Tmu_%s_%s_%s.nc' % (var, per, vi_stat)))[varn].sel(lat=slice(-17,-12), lon=slice(-75,-70)).mean(dim=['lon','lat'])
su_perc = xr.open_dataset(os.path.join(odir, 'Tsu_%s_%s_%s.nc' % (var, per, vi_stat)))[varn].sel(lat=slice(-17,-12), lon=slice(-75,-70)).mean(dim=['lon','lat'])
vi_perc = xr.open_dataset(os.path.join(odir, 'Tvi_%s_%s_%s.nc' % (var, per, vi_stat)))[varn].sel(lat=slice(-17,-12), lon=slice(-75,-70)).mean(dim=['lon','lat'])

In [ ]:
total=vi_perc+su_perc+mu_perc
pvi=vi_perc*100/total
psu=su_perc*100/total
pmu=mu_perc*100/total

In [ ]:
# INCERTIDUMBRE EN FRACCION
color_m = ['#1a4a8c', '#fb0b26', '#fbd326']
label = ['Incertidumbre Modelo', 'Incertidumbre Escenario', 'Variabilidad Interna']
fig, ax = plt.subplots(1, 1, figsize=(9, 6), sharey=True, sharex=True)
fig.subplots_adjust(hspace=0.17, wspace=0.05)
fig.suptitle('PERU', y=0.99, fontsize=18)
# PLOT
ax.stackplot(pmu['time.year'], pmu, psu, pvi, labels=label, colors=color_m)
ax.set_title('%s (%d-%d) - VI: %s' % (tri.upper(), y_ini, y_end, vi_stat), fontsize=12)
#
ax.set_ylabel('Porcentaje de incertidumbre (%)')
ax.legend(loc='upper left', fontsize=11)
ax.margins(0, 0)
ax.set_xlim(y_ini, y_end)
ax.set_xticks(year_ticks)
ax.set_ylim(0, 100)
plt.savefig(os.path.join(fdir, 'Fraccion_incertidumbre_peru_%s_%s_interpol_%s.png' % (var, tri, vi_stat)), dpi=100, bbox_inches='tight')